## OCR(Optical Character Recognition)
 - 이미지 속의 문자들을 컴퓨터가 인식 할 수 있도록 디지털 문자로 변환해 주는 기술 입니다.
   문서를 디지털화 하는 가장 기본적인 기술로 자리잡았으며, 이미지, pdf ,문서속 문자를 인식하고
   편집 및 검색이 가능하게 하는 AI기반의 기술을 의미합니다.

In [1]:
from dotenv import load_dotenv
import os

from langchain_openai.chat_models.base import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda

load_dotenv()
print(os.environ.get('OPENAI_API_KEY')[:20])

sk-proj-ktAbTJ1mXZ2V


## llm 준비

In [2]:
ocr_llm = ChatOPenAI(model="gpt-4o", temperature=0)

NameError: name 'ChatOPenAI' is not defined

In [ ]:
# base64

import base64

def encode_image(image_path):
    with open(image_path, "rb") as image:
        return base64.b64encode(image.read()).decode("utf-8")

In [ ]:
base_path = r"C:\back_0900_kyj\python\resource"

In [ ]:
receipt_name = r"reciept_01.png"
receipt_path = os.path.join(base_path, receipt_name)



## 프롬프트 준비

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("system", """
        You are an advanced vision model.

        Your goal is to extract ALL text from the image.

        IMPORTANT:
        - Do NOT miss any text
        - If text is unclear, infer the most likely correct text
        - Preserve all lines, even unusual ones
        - Do NOT drop promotional or "증정품" lines
        """),
            ("human", [
                {
                    "type": "text",
                    "text": """
        Extract ALL visible text from this receipt.

        Rules:
        - Include EVERY line (even if it looks unimportant)
        - Keep line breaks
        - Do NOT omit anything
        - If a word is unclear, guess the most likely correct word

        Output ONLY the extracted text.
        """
                },
                {
                    "type": "image_url",
                    "image_url": {
                        "url": "data:image/jpeg;base64,{encoded_image}"
                    }
                }
            ])
        ])

In [ ]:
ocr_chain = prompt | ocr_llm

In [ ]:
result = ocr_chain.invoke({
    "" , "상품의 목록만 json으로 반환해줘",
    "encode_image"
})

## JSON 형태로 변환

In [3]:
from typing import List
from pydantic import BaseModel, Field

In [4]:
class ReceiptItem(BaseModel):
    name: str = Field(description="The exact product name from the text. Copy verbatim.")
    count: int = Field(description="The quantity.")
    price: int = Field(description="The price text exactily as printed (e.g., '1,600' or '증정품').")

class ReceiptInfo(BaseModel):
    store_name: str = Field(description="Store name")
    date: str = Field(description="YYYY-MM-DD")
    items: List[ReceiptItem] = Field(description="List of purchansed items.")
    amount: int = Field(description="Discounted price")
    total_amount: int = Field(description="Price after discount")

In [ ]:
parse_llm =ChatOpenAI(model="gpt-4o-mini",temperature = 0)
# 리턴값은 무조건 ReceiptInfo타입
structured_llm = parse_llm.with_structured_output(ReciptInfo)



## 문제해결

In [6]:
parsing_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
        You are a deterministic parser.

        STRICT RULES:
        - Each line represents EXACTLY one item.
        - NEVER merge lines.
        - NEVER split lines.
        - NEVER infer or guess missing values.
        - Copy text exactly as written.

        You must process input line-by-line.
        """
            ),
            (
                "human",
                """
        Here is the raw receipt text:

        {receipt_text}

        Instructions:
        1. Only extract lines that match this pattern:
           [product name] [count] [price or '증정품']

        2. Ignore all other lines.

        3. Output must be JSON array:
        [
          {{"name": "...", "count": ..., "price": "..."}}
        ]

        4. Do not change numbers.
        5. Do not duplicate items.
        6. Do not create new items.

        Output JSON only.
        """
    )
])

## 2. 체인생성

In [ ]:
def transform_output(input_messages):
    return{"receipt_text": input_messages.content}

In [ ]:
parse_chain =parsing_prompt |

## 객체 탐지

In [ ]:
base_path = r"C:\back_0900_kyj\python\resource"

penguin_name = r"penguin.jpg"
cat_name = r"cat.jpg"



In [ ]:
ocr_llm = ChatOPenAI(model="gpt-4o", temperature=0)

In [ ]:
classifier_prompt = classifier_prompt | ocr_llm

In [7]:
cat_result = classifier_chain | ocr_llm

NameError: name 'classifier_chain' is not defined

In [ ]:
cat_result = classifier_chain.invoke({
    
})

In [ ]:
brief_prompt = ChatPromptTemplate.from_messages([
    ("system", """
        You are an expert visual scene analyst specification.

        Your task:
        - Analyze the provided image deeply and describe the scene in detail.

        Rules:
        - **Focus on:** Objects, actions, relationships, and precise colors (especially hair color, clothing color, accessories, and background colors).
        - **Detailed Breakdown:** Do not generalize. Identify specific entities and break down their appearance, posture, and environmental details.
        - **Length Rule:** Please provide a detailed explanation. The response must be at least 10 lines long. Ensure each logical point is separated by a line break to maintain readability.
        - **Constraint:** Do NOT guess unknown facts or speculate. Describe only what is visible.
        - **Language:** You must answer in Korean.
    """),
    ("human", [
        {
            "type": "text",
            "text": "Describe what is happening in this image in full detail, focusing on appearance, actions, colors, and relationships."
        },
        {
            "type": "image_url",
            "image_url": {
                "url": "data:image/jpeg;base64,{encoded_image}"
            }
        }
    ])
])

In [ ]:
brief_chain = brief_prompt | brief_llm

In [ ]:
yujung_name = "brief_yujung.png"
yujung_path = os.path.join(base_path,yujung_name)
base64_yujung = encode_image(yujung_path)

In [ ]:
breif_result = brief_chain.invoke({
    "endoed_image": base64_yujung
})

In [ ]:
print(brief_result.content)

## QA

In [ ]:
qa_prompt = ChatPromptTemplate.from_messages([
    ("system", """
        You are a question-answering assistant.

        Answer based ONLY on the given description.
        Do NOT assume anything beyond the description.
    """),
    ("human", """
        Description:
        {description}

        Question:
        {question}
    """)
    ])

In [ ]:
qa_chain = qa_prompt | brief_llm

In [ ]:
qa_result = qa_chain.invoke({
    "description" : brief_result.content,
    "question" : "남자가 입은 후드티의 색깔은 ?
})

In [8]:
qa_result

NameError: name 'qa_result' is not defined

In [ ]:
def extract_question(input_value):
    return input_value["question"]

def format_qa_input(input_value):
    return {
        "description" : input_value["description"]|.content
        "question":input_value["question"]

In [ ]:
final_qa_chain = (
    {
        "description" : brief_chain,
        "question" : RunnableLambda(extract_question)
    }
    | RunnableLambda(format_qa_input)
    | qa_chain
)

In [ ]:
final_qa_chain = ""